In [1]:
from sqlalchemy.ext.asyncio import create_async_engine

from sherloque.config import get_settings
from sherloque.index import PostgresIndex

In [2]:
get_settings().db_direct_async_str[:10], get_settings().db_direct_async_str[-10:]

('postgresql', '/sherloque')

In [3]:
engine = create_async_engine(get_settings().db_direct_async_str)
index = PostgresIndex(engine=engine, schema="public")

In [4]:
index

In [5]:
from sherloque import Sherloque
from sherloque.retrieve import BM25Retriever, VectorRetriever, BM25RetrieverConfig, VectorRetrieverConfig
from sherloque.model_providers import OpenRouterModelProvider
from sherloque.rank import CrossEncoderReRanker

In [6]:
bmc = BM25RetrieverConfig()
vc = VectorRetrieverConfig()

In [7]:
Sherloque?

Init signature:
Sherloque(
    index: sherloque.index.postgres.PostgresIndex,
    retrievers: list[sherloque.retrieve.base.BaseRetriever],
    rankers: list[sherloque.rank.cross_encoder_rerank.CrossEncoderReRanker],
)
Docstring:      <no docstring>
File:           ~/gen/sherloque/src/sherloque/__init__.py
Type:           type
Subclasses:     

In [8]:
OpenRouterModelProvider?

Init signature:
OpenRouterModelProvider(
    *,
    api_key: str | None = None,
    embed_model: str = 'openai/text-embedding-3-small',
    rerank_model: str = 'cohere/rerank-v3.5',
    base_url: str = 'https://openrouter.ai/api/v1',
    timeout: float | httpx.Timeout = 60.0,
    http_client: httpx.AsyncClient | None = None,
) -> None
Docstring:      Native OpenRouter embeddings and reranking API client.
File:           ~/gen/sherloque/src/sherloque/model_providers/openrouter.py
Type:           ABCMeta
Subclasses:     

In [9]:
ortr = OpenRouterModelProvider(api_key=get_settings().openrouter_api_key)

In [10]:
bmr = BM25Retriever(index=index, config=bmc)
vr = VectorRetriever(index=index, config=vc, model_client=ortr)

crs = CrossEncoderReRanker(engine=engine, model_client=ortr)

In [11]:
sh = Sherloque(index=index, retrievers=[bmr, vr], rankers=[crs])

In [12]:
res = await sh.retrieve(query="child tax credit")
res

[RetrieverResult(doc_id=1, doc_title='Child Tax Credit', score=0.01639344262295082),
 RetrieverResult(doc_id=104, doc_title='Get copies of patent, trade mark or design registration documents', score=0.01639344262295082),
 RetrieverResult(doc_id=126, doc_title='Tax credits if you leave or move to the UK', score=0.016129032258064516),
 RetrieverResult(doc_id=17, doc_title='Growing your business', score=0.016129032258064516),
 RetrieverResult(doc_id=631, doc_title='Benefits and prison', score=0.015873015873015872),
 RetrieverResult(doc_id=458, doc_title="Help if you're a student with a learning difficulty, health problem or disability", score=0.015873015873015872),
 RetrieverResult(doc_id=69, doc_title='Universal Credit', score=0.015625),
 RetrieverResult(doc_id=79, doc_title='Take goods temporarily out of the UK', score=0.015625),
 RetrieverResult(doc_id=2, doc_title='Working Tax Credit', score=0.015384615384615385),
 RetrieverResult(doc_id=315, doc_title='Employment contracts', score=0.

In [13]:
len(res)

20

In [14]:
await sh.search(query="employment law", limit=5)

[RetrieverResult(doc_id=316, doc_title='Employment status', score=0.66464967),
 RetrieverResult(doc_id=314, doc_title='Employers: preventing discrimination', score=0.29913127),
 RetrieverResult(doc_id=315, doc_title='Employment contracts', score=0.29407352),
 RetrieverResult(doc_id=162, doc_title='Appeal to the Employment Appeal Tribunal (EAT)', score=0.28140607),
 RetrieverResult(doc_id=205, doc_title='Discrimination: your rights', score=0.25740135)]

In [15]:
from typing import Any
from uuid import uuid4

from pydantic import BaseModel, Field

In [16]:
class Document(BaseModel):
    id: str = Field(default_factory=lambda: f"doc_{uuid4().hex}")
    text: str
    metadata: dict[str, Any] | None


In [19]:
BM25_SQL = (
    # @formatter:off
    """WITH q_tokens AS (
    SELECT t._id AS token_id, t.doc_freq
    FROM token t
    WHERE t.token = ANY(:tokens)
), stats AS (
    SELECT COUNT(*)::float AS total_docs, AVG(len) ::float AS avgdl
    FROM document
)
SELECT d.title AS doc_title, d.full_text AS doc_full_text, d._id as doc_id, SUM(
    LN(
        (stats.total_docs - q.doc_freq + 0.5) / (q.doc_freq + 0.5) + 1
    ) * (tds.tf * (:k1 + 1)) / (
        tds.tf + :k1 * (1 - :b + :b * d.len / stats.avgdl)
    )
) ::float AS score
FROM term_doc_stats tds
JOIN q_tokens q ON tds.token_id = q.token_id
JOIN document d ON tds.doc_id = d._id
CROSS JOIN stats
GROUP BY d._id
ORDER BY score DESC
LIMIT :top_k"""
    # @formatter:on
)


In [ ]:
_bm25_sql = text(
            BM25_SQL.format(**{
                # Escape colons for the text-bind parser after quoting identifiers.
                table.name: format_table(table).replace(":", r"\:")
                for table in (self._document, self._token, self._term_doc_stats)
            })
        )


In [22]:
from sqlalchemy import text

In [23]:
async with engine.connect() as conn:
    cur = await conn.execute(
        text(BM25_SQL),
        {"tokens": ["employ", "law"], "k1": 1.2, "b": 0.75, "top_k": 5},
    )
    rows = [record for record in cur]


In [ ]:
rows[0]

('UK visa sponsorship for employers', 'UK visa sponsorship for employers\n\nOverview\n\nYou’ll usually need a sponsor licence to employ someone to work for you from outside the UK. This in ... (23890 characters truncated) ... cence if you do not do this.\n\nChildren under 16\n\nYou must get a licence from the local education authority in the area where the child will work.', 556, 7.0681708047666145)

In [28]:
len(rows[0])

4

In [34]:
rows[0][2], rows[0][3]

(556, 7.0681708047666145)